# 🻀 rPPG-Toolbox: PhysFormer & SpikingPhysFormer on MMPD
### Multi-Temporal FusionStem + MediaPipe Dynamic Face Tracking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drozanes/rPPG-Toolbox/blob/main/MMPD_PhysFormer_SpikingPhysFormer_Colab.ipynb)

This notebook trains and evaluates **PhysFormer** and **SpikingPhysFormer** on the complete **MMPD** benchmark dataset (all 33 subjects, 660 raw video files) with:
1. **Multi-Temporal FusionStem** (RhythmFormer CVPR 2024): Filters mobile rolling shutter and compression noise across -1, t-2, t-4$ frame differences.
2. **MediaPipe BlazeFace Dynamic Face Tracking**: Continuous landmark-guided face tracking that eliminates bounding box jumping artifacts during head movements.

### ⚙️ Recommended Colab Runtime Setup:
- **Hardware Accelerator:**  (efficient: ~1.5 - 2 CU/hr) or 
- **High-RAM:** **Toggle ON** (provides 53.5 GB RAM and 8 vCPUs for rapid data streaming)

### Key Features:
- **Zero Preprocessing Footprint:** Uses online on-the-fly streaming directly from compressed  files without needing 940 GB disk extraction.
- **Real-Time Google Drive Backup:** Checkpoints and benchmark database are mirrored to  after every epoch.
- **Weights & Biases Logging:** Real-time curves for training loss, validation loss, learning rate, and test evaluation metrics.
- **Independent Execution:** Train PhysFormer, SpikingPhysFormer, or both in dedicated cells.

In [ ]:
# 1. Verify GPU Allocation & System Resources
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


In [ ]:
# 2. (Highly Recommended) Mount Google Drive for Real-Time Model Saving
from google.colab import drive
import os
drive.mount("/content/drive")

backup_dir = "/content/drive/MyDrive/rPPG_Checkpoints"
os.makedirs(backup_dir, exist_ok=True)
print(f"✅ Google Drive mounted! Checkpoints will be mirrored to: {backup_dir}")


In [ ]:
# 3. Clone Repository and Install Dependencies
!git clone https://github.com/drozanes/rPPG-Toolbox.git
%cd rPPG-Toolbox

# Install dependencies including MediaPipe, SpikingJelly (for SpikingPhysFormer), and hf-transfer
!pip install -q yacs mat73 huggingface_hub wandb pyyaml scipy opencv-python tqdm pandas matplotlib scikit-learn hf-transfer mediapipe spikingjelly


In [ ]:
# 4. Disk Space Verification & Proactive Cleanup
import shutil
total, used, free = shutil.disk_usage("/")
print(f"Colab Root Disk: Total = {total/(1024**3):.1f} GB, Usable Free = {free/(1024**3):.1f} GB")

if free / (1024**3) < 55:
    print("Cleaning package cache for extra disk headroom...")
    !apt-get clean > /dev/null 2>&1
    !rm -rf /root/.cache/pip
    _, _, free = shutil.disk_usage("/")
    print(f"Updated Free Disk: {free/(1024**3):.1f} GB")


In [ ]:
# 5. Download Raw MMPD Mini Dataset (~47.3 GB, 660 files) from Hugging Face
import os, glob, getpass
from huggingface_hub import snapshot_download

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    print("Please enter your Hugging Face Access Token (read access to WeiQian98/mini_MMPD):")
    HF_TOKEN = getpass.getpass("HF Token: ").strip()

REPO_ID = "WeiQian98/mini_MMPD"
data_dir = "./MMPD_dataset"
os.makedirs(data_dir, exist_ok=True)

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
print(f"Downloading all 660 .mat files from {REPO_ID}...")
snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=data_dir,
    token=HF_TOKEN,
    ignore_patterns=[".DS_Store"],
    max_workers=8
)

mat_files = glob.glob(f"{data_dir}/**/*.mat", recursive=True)
print(f"✅ Download verification: {len(mat_files)} / 660 .mat files successfully downloaded!")
assert len(mat_files) == 660, f"Expected 660 files, found {len(mat_files)}"


In [ ]:
# 6. Log in to Weights & Biases for live metrics tracking
import wandb
wandb.login()


---
## 🔥 Experiment 1: PhysFormer (FusionStem + MediaPipe Dynamic Tracking)
Spatiotemporal Vision Transformer (12 layers, 160 frames) with multi-temporal difference fusion.

In [ ]:
# 7A. Train PhysFormer (FusionStem + MediaPipe) on MMPD
!python main.py --config_file configs/train_configs/MMPD_PHYSFORMER_FUSIONSTEM_MEDIAPIPE_ONLINE.yaml --device cuda --num_workers 4 --backup_dir /content/drive/MyDrive/rPPG_Checkpoints


In [ ]:
# 7B. (Optional / Standalone) Evaluate PhysFormer Checkpoint on Test Set
# !python main.py --config_file configs/train_configs/MMPD_PHYSFORMER_FUSIONSTEM_MEDIAPIPE_ONLINE.yaml --device cuda --num_workers 4 --toolbox_mode only_test --model_path MMPD_PhysFormer_FusionStem_MediaPipe_Online_Epoch15.pth --backup_dir /content/drive/MyDrive/rPPG_Checkpoints


---
## ⚡ Experiment 2: SpikingPhysFormer (FusionStem + MediaPipe Dynamic Tracking)
4-layer Spiking Neural Network (SNN) + Temporal Transformer with Multi-Temporal FusionStem.

In [ ]:
# 8A. Train SpikingPhysFormer (FusionStem + MediaPipe) on MMPD
!python main.py --config_file configs/train_configs/MMPD_SPIKINGPHYSFORMER_FUSIONSTEM_MEDIAPIPE_ONLINE.yaml --device cuda --num_workers 4 --backup_dir /content/drive/MyDrive/rPPG_Checkpoints


In [ ]:
# 8B. (Optional / Standalone) Evaluate SpikingPhysFormer Checkpoint on Test Set
# !python main.py --config_file configs/train_configs/MMPD_SPIKINGPHYSFORMER_FUSIONSTEM_MEDIAPIPE_ONLINE.yaml --device cuda --num_workers 4 --toolbox_mode only_test --model_path MMPD_SpikingPhysFormer_FusionStem_MediaPipe_Online_Epoch10.pth --backup_dir /content/drive/MyDrive/rPPG_Checkpoints


---
## 📊 Results & Checkpoint Downloads

In [ ]:
# 9. Display Updated Benchmark Results
import os, pandas as pd
if os.path.exists("rPPG_results_database.csv"):
    df = pd.read_csv("rPPG_results_database.csv")
    print("
=== Benchmark Database (MMPD Entries) ===")
    mmpd_df = df[df["Dataset"] == "MMPD"]
    display(mmpd_df)


In [ ]:
# 10. Bundle Trained Checkpoints and Download Zip Archive
import glob, shutil
from google.colab import files

!zip -r MMPD_Transformer_Checkpoints.zip runs/exp/*/PreTrainedModels/*.pth rPPG_results_database.csv
files.download("MMPD_Transformer_Checkpoints.zip")
